# Chapter 3 — Visualizing Quantum Programs with the QDK

**Learning objectives**
- Generate and render circuit diagrams with `qsharp.circuit()` and the `Circuit` widget
- Choose the right circuit generation method: synthesis vs simulation
- Understand how target profile affects circuit diagrams
- Visualize quantum state with `DumpMachine()` and `DumpRegister()`
- Plot measurement histograms with `qsharp_widgets.Histogram` and matplotlib
- Inspect gate decompositions visually for debugging

## Setup

In [ ]:
import qsharp
from qsharp import CircuitGenerationMethod
from qsharp_widgets import Circuit, Histogram
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

print(f"qsharp {__import__('importlib.metadata', fromlist=['version']).version('qsharp')}")

## 3.1 Circuit Diagrams: Basic Usage

`qsharp.circuit(expr)` synthesizes a circuit diagram for any Q# entry expression. Wrap it in `Circuit()` to get an interactive SVG rendering in Jupyter.

Pass a concrete call expression rather than an operation name when your operation takes `Qubit[]` arguments.

In [ ]:
%%qsharp

operation GHZ(n : Int) : Result[] {
    use qs = Qubit[n];
    H(qs[0]);
    for i in 1..n-1 {
        CNOT(qs[0], qs[i]);
    }
    MResetEachZ(qs)
}

In [ ]:
# Render a 4-qubit GHZ circuit
Circuit(qsharp.circuit("GHZ(4)"))

In [ ]:
%%qsharp

operation BellState() : Result[] {
    use q = Qubit[2];
    H(q[0]);
    CNOT(q[0], q[1]);
    MResetEachZ(q)
}

In [ ]:
Circuit(qsharp.circuit("BellState()"))

## 3.2 Circuit Diagrams for Operations (Not Entry Expressions)

If you want a circuit for a reusable operation that takes qubits as arguments (rather than allocating them internally), use `qsharp.circuit(operation="OpName")`. For `Qubit[]` arguments the diagram defaults to showing 2 wires — use a wrapper with a concrete qubit count to override.

In [ ]:
%%qsharp

import Std.Convert.IntAsDouble;

operation ApplyQFT(qs : Qubit[]) : Unit is Adj + Ctl {
    let n = Length(qs);
    for i in 0..n-1 {
        H(qs[i]);
        for j in i+1..n-1 {
            Controlled R1([qs[j]], (2.0 * Microsoft.Quantum.Math.PI() / IntAsDouble(2^(j-i+1)), qs[i]));
        }
    }
    // Swap to bit-reverse order
    for i in 0..n/2-1 {
        SWAP(qs[i], qs[n-1-i]);
    }
}

// Wrapper with a fixed qubit count for the circuit diagram
operation QFT4() : Unit {
    use qs = Qubit[4];
    ApplyQFT(qs);
    ResetAll(qs);
}

In [ ]:
# Diagram via entry expression (concrete qubit count)
Circuit(qsharp.circuit("QFT4()"))

## 3.3 Mid-Circuit Measurements: GenerationMethod.Simulate

Circuits with mid-circuit measurements and classical branching can't be statically synthesized — the circuit structure depends on measurement outcomes. Use `generation_method=CircuitGenerationMethod.Simulate` to run the circuit once and record the actual path taken.

The resulting diagram shows one concrete execution trace.

In [ ]:
%%qsharp

operation TeleportVisualize() : Result {
    use msg = Qubit();
    use here = Qubit();
    use there = Qubit();

    // Prepare message in |+⟩
    H(msg);

    // Create Bell pair
    H(here);
    CNOT(here, there);

    // Bell measurement
    CNOT(msg, here);
    H(msg);
    let m1 = M(msg);
    let m2 = M(here);

    // Classical correction
    if m2 == One { X(there); }
    if m1 == One { Z(there); }

    MResetZ(there)
}

In [ ]:
# Static synthesis fails for mid-circuit measurement + classical control
try:
    Circuit(qsharp.circuit("TeleportVisualize()"))
except qsharp.QSharpError as e:
    print(f"Static synthesis failed (expected): {e}")

In [ ]:
# Simulation-based: records one actual execution path
Circuit(qsharp.circuit("TeleportVisualize()", generation_method=CircuitGenerationMethod.Simulate))

## 3.4 Profile-Aware Circuit Synthesis

The circuit diagram reflects the current target profile. Under `TargetProfile.Base`, high-level operations are decomposed to native gates — the diagram shows the actual gate sequence that will run on hardware. This is useful for understanding gate counts and circuit depth.

In [ ]:
%%qsharp

operation GroverDiffusion(qs : Qubit[]) : Unit {
    ApplyToEach(H, qs);
    ApplyToEach(X, qs);
    Controlled Z(qs[0..Length(qs)-2], qs[Length(qs)-1]);
    ApplyToEach(X, qs);
    ApplyToEach(H, qs);
}

operation GroverStep3() : Unit {
    use qs = Qubit[3];
    GroverDiffusion(qs);
    ResetAll(qs);
}

In [ ]:
# Unrestricted profile: high-level view
print("Unrestricted profile (high-level):")
Circuit(qsharp.circuit("GroverStep3()"))

In [ ]:
# Switch to Base profile: gates will be decomposed as for hardware submission
qsharp.init(target_profile=qsharp.TargetProfile.Base)

In [ ]:
%%qsharp

operation GroverDiffusion(qs : Qubit[]) : Unit {
    ApplyToEach(H, qs);
    ApplyToEach(X, qs);
    Controlled Z(qs[0..Length(qs)-2], qs[Length(qs)-1]);
    ApplyToEach(X, qs);
    ApplyToEach(H, qs);
}

operation GroverStep3() : Unit {
    use qs = Qubit[3];
    GroverDiffusion(qs);
    ResetAll(qs);
}

In [ ]:
print("Base profile (gate-decomposed for hardware):")
Circuit(qsharp.circuit("GroverStep3()"))

In [ ]:
# Reset to Unrestricted for the rest of the chapter
qsharp.init(target_profile=qsharp.TargetProfile.Unrestricted)

In [ ]:
%%qsharp

import Std.Diagnostics.DumpMachine;

operation BellState() : Result[] {
    use q = Qubit[2];
    H(q[0]);
    CNOT(q[0], q[1]);
    MResetEachZ(q)
}

operation GHZ(n : Int) : Result[] {
    use qs = Qubit[n];
    H(qs[0]);
    for i in 1..n-1 {
        CNOT(qs[0], qs[i]);
    }
    MResetEachZ(qs)
}

## 3.5 State Visualization with DumpMachine

`DumpMachine()` prints the full amplitude vector. In Jupyter with `qsharp_widgets` installed, it renders as an interactive HTML table showing amplitudes, phases, and probabilities. `DumpRegister(qs)` isolates a subsystem (valid only when the subsystem is unentangled with the rest).

In [ ]:
%%qsharp

import Std.Diagnostics.*;

operation ShowSuperposition() : Unit {
    use q = Qubit();
    Message("Initial |0> state:");
    DumpMachine();

    H(q);
    Message("After H - equal superposition |+>:");
    DumpMachine();

    T(q);
    Message("After T gate - phase rotation:");
    DumpMachine();

    Reset(q);
}

ShowSuperposition()

In [ ]:
%%qsharp

import Std.Diagnostics.*;

operation InspectBell() : Unit {
    use q = Qubit[2];
    H(q[0]);

    Message("Separable: q[0] in |+>, q[1] in |0> - DumpRegister works:");
    DumpRegister(q[0..0]);

    CNOT(q[0], q[1]);
    Message("Entangled Bell state - full machine:");
    DumpMachine();

    ResetAll(q);
}

InspectBell()

In [ ]:
%%qsharp

import Std.Diagnostics.DumpMachine;

// Phase kickback: controlled-Z acting on |+>|1> kicks phase onto control
operation PhaseKickback() : Unit {
    use control = Qubit();
    use target = Qubit();

    H(control);
    X(target);

    Message("Before CZ:");
    DumpMachine();

    Controlled Z([control], target);

    Message("After CZ - phase kicked back onto control:");
    DumpMachine();

    ResetAll([control, target]);
}

PhaseKickback()

## 3.6 Histogram Visualization

The `Histogram` widget from `qsharp_widgets` renders measurement result distributions as an interactive bar chart — more readable than raw Python dicts for multi-qubit outcomes.

In [ ]:
# Bell state should produce only 00 and 11
results = qsharp.run("BellState()", shots=500)
Histogram(results)

In [ ]:
# 4-qubit GHZ: should produce only 0000 and 1111
results_ghz = qsharp.run("GHZ(4)", shots=500)
Histogram(results_ghz)

In [ ]:
%%qsharp

// Uniform superposition over all 3-qubit basis states
operation Uniform3() : Result[] {
    use qs = Qubit[3];
    ApplyToEach(H, qs);
    MResetEachZ(qs)
}

In [ ]:
# All 8 outcomes should appear with roughly equal probability
results_uniform = qsharp.run("Uniform3()", shots=800)
Histogram(results_uniform)

## 3.7 Custom Matplotlib Plots

For publication-quality figures or custom analysis, convert results to a `Counter` and plot with matplotlib.

In [ ]:
%%qsharp

operation BellFidelityTest() : Result[] {
    use q = Qubit[2];
    H(q[0]);
    CNOT(q[0], q[1]);
    MResetEachZ(q)
}

In [ ]:
# Plot Bell state fidelity vs depolarizing noise level
noise_levels = np.linspace(0, 0.2, 12)
fidelities = []

for p in noise_levels:
    if p == 0:
        shots = qsharp.run("BellFidelityTest()", 400)
    else:
        shots = qsharp.run("BellFidelityTest()", 400, noise=qsharp.DepolarizingNoise(p))
    correlated = sum(1 for r in shots if r[0] == r[1]) / len(shots)
    fidelities.append(correlated)

plt.figure(figsize=(7, 4))
plt.plot(noise_levels, fidelities, 'o-', color='steelblue')
plt.axhline(0.5, color='red', linestyle='--', label='Classical limit')
plt.xlabel("Depolarizing noise probability p")
plt.ylabel("Bell state fidelity")
plt.title("Bell State Fidelity vs Depolarizing Noise")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
%%qsharp

operation QFTCircuit(n : Int) : Result[] {
    use qs = Qubit[n];
    for i in 0..n-1 {
        H(qs[i]);
    }
    MResetEachZ(qs)
}

In [ ]:
import time

qubit_counts = list(range(2, 18, 2))
runtimes = []

for n in qubit_counts:
    start = time.perf_counter()
    qsharp.run(f"QFTCircuit({n})", shots=50)
    runtimes.append((time.perf_counter() - start) * 1000)

plt.figure(figsize=(7, 4))
plt.plot(qubit_counts, runtimes, 's-', color='darkorange')
plt.xlabel("Qubit count")
plt.ylabel("Runtime (ms, 50 shots)")
plt.title("Simulation Runtime vs Qubit Count")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3.8 Using Circuits for Debugging

Circuit diagrams are a practical debugging tool. If a Q# operation produces unexpected results, rendering it as a circuit helps you spot wrong gate orderings, missing adjoint gates, or control qubit mistakes before running expensive simulations.

In [ ]:
%%qsharp

// Correct: applies X then undoes it via Adjoint — net identity
operation RoundTrip() : Result {
    use q = Qubit();
    X(q);
    Adjoint X(q);
    MResetZ(q)
}

// Bug: CNOT applied before H instead of after
operation BuggyEntangle() : Result[] {
    use q = Qubit[2];
    CNOT(q[0], q[1]);
    H(q[0]);
    MResetEachZ(q)
}

In [ ]:
print("Round-trip circuit (should be identity):")
Circuit(qsharp.circuit("RoundTrip()"))

In [ ]:
print("Buggy entanglement circuit — CNOT before H instead of after:")
Circuit(qsharp.circuit("BuggyEntangle()"))

In [ ]:
# Confirm the buggy version produces wrong distribution (not a Bell state)
correct_results = qsharp.run("BellState()", shots=400)
buggy_results = qsharp.run("BuggyEntangle()", shots=400)

print("Correct Bell state outcomes:", Counter(str(r) for r in correct_results).most_common())
print("Buggy entangle outcomes:    ", Counter(str(r) for r in buggy_results).most_common())

## Summary

- `qsharp.circuit(expr)` synthesizes a circuit diagram; wrap in `Circuit()` for interactive SVG rendering
- Use `generation_method=CircuitGenerationMethod.Simulate` for circuits with mid-circuit measurements or classical control flow
- `qsharp.circuit(operation="OpName")` generates a diagram for a reusable operation; use concrete-qubit wrapper operations for `Qubit[]` args
- Target profile matters: `Base` shows gate-decomposed circuits as they'd run on hardware; `Unrestricted` shows high-level operations
- `DumpMachine()` / `DumpRegister()` inspect quantum state interactively in Jupyter
- `Histogram(results)` renders measurement distributions; matplotlib gives full control for custom plots
- Circuit diagrams are a fast first-pass debugging tool to catch gate ordering and control qubit mistakes